# Chapter 1 - What Is Generative AI?
## Hands-On Colab Lab for Network Engineers

**9 progressive demos - everything from first API call to security best practices:**

| # | Demo | What it shows |
|---|------|---------------|
| 1 | Setup and first API call | 5 lines of Python to query an LLM |
| 2 | Tokenisation | How text becomes numbers - and why IP addresses cost more tokens |
| 3 | Temperature and Softmax | Why LLMs are probabilistic and how to control it |
| 4 | Self-Attention in pure NumPy | The transformer core, built from scratch |
| 5 | Embeddings and Cosine Similarity | Why "interface down" and "link failure" are mathematically close |
| 6 | Config review | AI as your senior engineer |
| 7 | Regex vs AI parsing | Same task, two approaches - when to use which |
| 8 | Context window and token cost | Budget tokens like bandwidth |
| 9 | Security: Sanitise before sending | Strip credentials before sending to any API |

~25 minutes | ~USD 0.10 in API calls | Runs fully in Google Colab

> Tip: Run cells top-to-bottom. Every cell is self-contained and heavily commented.


---
## Demo 1 - Setup and Your First API Call

Install the Anthropic SDK and make the simplest possible call.
Think of this as your first ping to the AI.


In [ ]:
# Install the Anthropic Python SDK (only needed once per Colab session)
\!pip install -q anthropic
print("Anthropic SDK installed successfully")


In [ ]:
import os, json, re
import numpy as np
from anthropic import Anthropic

# Load your API key
# Option A: Colab Secrets (recommended)
#   Click the key icon in the left sidebar and add ANTHROPIC_API_KEY
# Option B: paste directly (never commit this to git\!)
try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    print("API key loaded from Colab Secrets")
except Exception:
    os.environ["ANTHROPIC_API_KEY"] = input("Paste your Anthropic API key: ")

client = Anthropic()
MODEL  = "claude-haiku-4-5-20251001"   # Cheapest model - perfect for demos
print(f"Client ready. Model: {MODEL}")


In [ ]:
# The simplest possible LLM call - only 5 lines of Python
response = client.messages.create(
    model=MODEL,
    max_tokens=300,
    messages=[{
        "role": "user",
        "content": "Explain OSPF in 3 sentences for a junior network engineer."
    }]
)

print("=== Claude answer ===")
print(response.content[0].text)
print()
print(f"Input tokens  : {response.usage.input_tokens}")
print(f"Output tokens : {response.usage.output_tokens}")
print(f"Total tokens  : {response.usage.input_tokens + response.usage.output_tokens}")
print()
print("This is all it takes. No server. No ML expertise. Just an HTTPS call.")


---
## Demo 2 - Tokenisation: How Text Becomes Numbers

Before an LLM processes your router config, it converts text to **tokens**.
A token is roughly 3-4 characters (about 0.75 words).

**Networking analogy:** Tokens are to LLMs what packets are to networks.
The context window (200K for Claude) is the bandwidth budget per API call.

**Key facts:**
- "192.168.1.1" = 7 tokens (each dot is its own token)
- "GigabitEthernet0/0/0" = about 6 tokens
- A 500-line router config = ~2,000 tokens = ~USD 0.006 with Sonnet


In [ ]:
# Build a word-level tokeniser from scratch to see exactly what happens
# Real LLMs use Byte-Pair Encoding (BPE) - more efficient but the same idea

def simple_tokenise(text):
    """Split text into tokens. Real BPE tokenisers also handle subwords."""
    tokens = re.split(r"(\s+|[,./\!?;:\-])", text)
    return [t for t in tokens if t.strip()]

examples = [
    "show ip interface brief",
    "192.168.1.1",
    "GigabitEthernet0/0/0",
    "router ospf 1",
    "neighbor 10.0.0.1 remote-as 65001",
]

print(f"  {'Text':<35}  {'Tokens':}")
print("-" * 80)
for text in examples:
    toks = simple_tokenise(text)
    print(f"  {text:<35}  {toks}  (count: {len(toks)})")

print()
print("Notice: 192.168.1.1 becomes 7 tokens because each dot is a separate token.")
print("This is why IP addresses and interface names cost more than plain English words.")


In [ ]:
# Build a vocabulary: mapping each unique token to an integer ID
# The LLM embedding layer converts integer IDs into vectors

sample_config_text = """
interface GigabitEthernet0/0
 description Uplink to ISP
 ip address 203.0.113.1 255.255.255.252
 no shutdown
router ospf 1
 router-id 192.168.1.1
 network 10.0.0.0 0.0.255.255 area 0
"""

all_tokens    = simple_tokenise(sample_config_text)
unique_tokens = sorted(set(all_tokens))
vocab         = {t: i for i, t in enumerate(unique_tokens)}

print(f"Total tokens    : {len(all_tokens)}")
print(f"Unique (vocab)  : {len(unique_tokens)}")
print()
print("Vocabulary sample (token -> ID):")
for tok, idx in list(vocab.items())[:12]:
    print(f"  {repr(tok):<25} -> {idx}")
print()
first = "interface GigabitEthernet0/0"
ids = [vocab.get(t, -1) for t in simple_tokenise(first)]
print(f"Line: '{first}'")
print(f"Becomes token IDs: {ids}")
print("This is what the model receives - integers, not text")


In [ ]:
# Count tokens with the official Claude API before sending large configs
# Like a dry-run to check bandwidth before a large file transfer

configs = {
    "Simple question":   "What is OSPF?",
    "One show command":  "show ip interface brief",
    "8-line config":     sample_config_text.strip(),
}

print(f"{'Label':<25} {'Tokens':>8}  {'Haiku cost':>12}  {'Sonnet cost':>12}")
print("-" * 65)
for label, text in configs.items():
    count  = client.messages.count_tokens(model=MODEL, messages=[{"role":"user","content":text}])
    tokens = count.input_tokens
    cost_h = tokens * 0.0000008
    cost_s = tokens * 0.000003
    print(f"{label:<25} {tokens:>8}    ")

print()
print("Rule: a 500-line config ~= 2,000 tokens ~= USD 0.006 with Sonnet")
print("Think of tokens as bytes and the context window as your MTU")


---
## Demo 3 - Temperature and Softmax: Controlling Randomness

An LLM outputs a **probability distribution** over its vocabulary at each step.
**Softmax** converts raw scores (logits) into probabilities that sum to 1.0.
**Temperature** controls how peaked or flat that distribution is.

**Networking analogy:** Temperature is like jitter in a QoS queue.
- Temperature 0.0 = deterministic, identical output every time
- Temperature 1.0 = natural variation
- Temperature > 1.5 = creative and sometimes unpredictable

Practical guidance:
- Config generation, compliance checks, structured parsing: temperature 0.0-0.3
- Writing documentation, brainstorming, summarising: temperature 0.7-1.0


In [ ]:
import numpy as np

def softmax(logits, temperature=1.0):
    """Convert raw model scores (logits) into a probability distribution.

    Low temperature  = peaked distribution = model is confident and consistent
    High temperature = flat distribution  = model is creative and random
    """
    scaled = np.array(logits, dtype=float) / temperature
    scaled -= scaled.max()       # subtract max for numerical stability
    exp_v  = np.exp(scaled)
    return exp_v / exp_v.sum()

# Scenario: model just read "router ospf" and is picking the next token
# Three candidates with their raw scores:
candidates   = ["1", "100", "65001"]
logits       = [3.5,  1.2,   0.3 ]   # "1" is most common in real configs
temperatures = [0.1, 0.5, 1.0, 2.0]

print("Probability of each token appearing after 'router ospf':")
print(f"  {'Token':<10} {'Raw score':>10}   Temp=0.1   Temp=0.5   Temp=1.0   Temp=2.0")
print("-" * 72)
for i, (tok, score) in enumerate(zip(candidates, logits)):
    probs = [softmax(logits, t)[i] for t in temperatures]
    ps    = "".join(f"   {p:.4f}" for p in probs)
    print(f"  {tok:<10} {score:>10.1f}  {ps}")

print()
print("Temperature 0.1: token 1 wins almost every time (good for config generation)")
print("Temperature 2.0: all tokens have nearly equal probability (creative mode)")
print("Use temperature 0.0 to 0.3 for anything that requires deterministic output")


In [ ]:
# See temperature in action with real API calls
prompt = "In one sentence, describe the best OSPF area design for a large enterprise."

print("Same prompt, three different temperatures:")
print()
for temp in [0.0, 0.5, 1.5]:
    resp = client.messages.create(
        model=MODEL,
        max_tokens=80,
        temperature=temp,
        messages=[{"role": "user", "content": prompt}]
    )
    print(f"Temperature {temp:.1f}:")
    print(f"  {resp.content[0].text.strip()[:250]}")
    print()

print("Run this cell multiple times. Temperature 0.0 gives the same answer every time.")
print("Temperature 1.5 gives a different answer each run.")


---
## Demo 4 - Self-Attention: How the Model Understands Context

Transformers process all tokens simultaneously (not word by word).
**Self-Attention** is the mechanism that lets every token look at every other
token and compute how much it should pay attention to each one.

**Networking analogy:** Self-attention is like a Route Reflector (RR).
It does not treat all peers equally - it computes a relevance weight for each peer.
So the token "OSPF" attends strongly to "area" and "cost" even when they appear
10 lines apart in a config file.

We build a simplified version using **pure NumPy** - no ML framework needed.


In [ ]:
import numpy as np

# Five networking tokens, each with a hand-crafted 4D embedding
# In a real transformer these are learned vectors with thousands of dimensions
tokens = ["OSPF", "area", "cost", "interface", "BGP"]

np.random.seed(42)
embeddings = np.array([
    [0.9, 0.1, 0.8, 0.2],  # OSPF      - interior routing protocol
    [0.8, 0.2, 0.7, 0.3],  # area      - core OSPF concept (close to OSPF vector)
    [0.7, 0.3, 0.6, 0.4],  # cost      - OSPF metric     (close to OSPF vector)
    [0.1, 0.9, 0.1, 0.8],  # interface - physical layer  (far from OSPF vector)
    [0.2, 0.8, 0.2, 0.7],  # BGP       - exterior protocol (far from OSPF vector)
])

# Step 1: Attention scores = dot product of every pair
# High dot product = vectors point in same direction = semantically related
attn_scores = embeddings @ embeddings.T   # shape (5, 5)

# Step 2: Convert to weights with softmax so each row sums to 1.0
def softmax2d(x):
    e = np.exp(x - x.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)

attn_w = softmax2d(attn_scores)   # shape (5, 5)

# Step 3: New context-aware representations = weighted average of all embeddings
ctx = attn_w @ embeddings   # shape (5, 4)

# Visualise: how does OSPF (row 0) attend to each token?
print("How much does the OSPF token attend to each other token?")
print()
for tok, w in zip(tokens, attn_w[0]):
    bar = chr(9608) * int(w * 50)
    print(f"  {tok:<12} {w:.4f}  {bar}")

print()
print("OSPF attends most strongly to area and cost (same routing protocol family)")
print("OSPF attends least to BGP and interface (different technical domains)")
print()
print("This attention mechanism is why the model correctly links OSPF with cost")
print("even when those words appear on lines 1 and 20 of a long config file.")


---
## Demo 5 - Embeddings and Cosine Similarity: Meaning as Math

An **embedding** is a list of numbers that captures the meaning of a text string.
**Cosine similarity** measures how similar two embeddings are:
- 1.0 = identical meaning
- 0.0 = completely unrelated
- -1.0 = opposite meaning

This is the mathematical core of:
- RAG systems (find the most relevant docs for a query)
- Semantic search in your internal knowledge base
- Alert deduplication in SIEM (same event, different wording)

**Networking analogy:** Embeddings are like OSPF LSAs - they encode the topology
of meaning. Cosine similarity is like checking if two routers share the same OSPF area.


In [ ]:
def cosine_sim(a, b):
    """Cosine similarity = dot(a,b) / (|a| * |b|)
    1.0 = same meaning | 0.0 = unrelated | -1.0 = opposite
    """
    d = np.dot(a, b)
    return d / (np.linalg.norm(a) * np.linalg.norm(b)) if np.linalg.norm(a) and np.linalg.norm(b) else 0.0

# Six-dimensional meaning vectors for common network events
# Dimensions: [routing, switching, security, config, monitoring, protocol]
concept_vecs = {
    "OSPF route leak":         np.array([0.9, 0.1, 0.6, 0.8, 0.3, 0.9]),
    "BGP misconfiguration":    np.array([0.8, 0.1, 0.5, 0.9, 0.2, 0.8]),
    "interface flapping":      np.array([0.5, 0.7, 0.2, 0.3, 0.8, 0.4]),
    "link up/down alerts":     np.array([0.4, 0.8, 0.2, 0.2, 0.9, 0.3]),
    "firewall rule violation": np.array([0.1, 0.2, 0.9, 0.7, 0.5, 0.2]),
    "ACL misconfiguration":    np.array([0.2, 0.1, 0.9, 0.8, 0.4, 0.3]),
}

concepts = list(concept_vecs.keys())
vectors  = list(concept_vecs.values())
short    = [c[:13] for c in concepts]

print("Cosine Similarity Matrix (1.0 = same meaning, 0.0 = unrelated):
")
print(f"  {'Concept':<28}" + "".join(f"{n:>14}" for n in short))
print("-" * 112)
for c_a, v_a in zip(concepts, vectors):
    row = f"  {c_a:<28}"
    for v_b in vectors:
        row += f"{cosine_sim(v_a, v_b):>14.3f}"
    print(row)

print()
print("OSPF route leak  <->  BGP misconfiguration   : HIGH - both routing problems")
print("interface flapping  <->  link up/down alerts  : HIGH - same event, different words")
print("OSPF route leak  <->  firewall rule violation : LOW  - different technical domain")
print()
print("RAG systems work by embedding your question, comparing it to all document")
print("embeddings, and retrieving the top-N most similar documents as context.")


---
## Demo 6 - Config Review: AI as Your Senior Engineer

This is the demo most relevant to your daily operations.
Paste any router config and get a structured security and best-practice review.

No custom rules to write. No vendor-specific scripts to maintain.
Just describe what you want in plain English.


In [ ]:
# Router config with several intentional issues - can you spot them first?
sample_config = """
hostname core-router-01
!
enable password cisco123
!
interface Loopback0
 ip address 192.168.1.1 255.255.255.255
!
interface GigabitEthernet0/0
 description Uplink to ISP-A
 ip address 203.0.113.1 255.255.255.252
 no shutdown
!
interface GigabitEthernet0/1
 description Trunk to DC-SPINE-01
 ip address 10.0.1.1 255.255.255.0
 shutdown
!
router ospf 1
 router-id 192.168.1.1
 network 10.0.0.0 0.0.255.255 area 0
 network 203.0.113.0 0.0.0.255 area 0
!
ip route 0.0.0.0 0.0.0.0 203.0.113.2
!
snmp-server community public RO
snmp-server community private RW
!
line vty 0 4
 transport input telnet
 no login
"""

response = client.messages.create(
    model=MODEL,
    max_tokens=1500,
    system=(
        "You are a senior network security engineer doing a formal config review. "
        "Always reference the exact config line that causes a concern. "
        "Structure your response in exactly three sections: "
        "CRITICAL ISSUES, BEST PRACTICE VIOLATIONS, WHAT IS DONE WELL."
    ),
    messages=[{
        "role": "user",
        "content": f"Please review this router configuration:

{sample_config}"
    }]
)

print(response.content[0].text)
print()
print(f"Input tokens  : {response.usage.input_tokens}")
print(f"Output tokens : {response.usage.output_tokens}")
cost = (response.usage.input_tokens * 0.0000008) + (response.usage.output_tokens * 0.000004)
print(f"Estimated cost: USD {cost:.5f} (Haiku pricing)")
print()
print("A human senior engineer takes 15-20 minutes for this review. AI takes seconds.")


---
## Demo 7 - Traditional Automation vs Generative AI

Same task: extract all interfaces from a router config.

- **Regex**: fast, free, deterministic - works for ONE vendor only
- **AI**: slower, costs tokens - works for ANY vendor syntax

**Networking analogy:** Static route vs routing protocol.
A static route is fast and predictable but only works for one specific destination.
A routing protocol adapts automatically to topology changes - just like AI adapts to any vendor.

**Decision guide:**
- Single vendor + must be 100% deterministic + no API budget: use regex
- Multi-vendor + unknown format + need flexibility: use AI


In [ ]:
import re, json

# APPROACH 1: Traditional regex
# Pros: instant, free, zero network calls, completely deterministic
# Cons: breaks completely on NX-OS, JunOS, EOS, or non-standard IOS

def parse_ifaces_regex(config):
    """Parse Cisco IOS interfaces. Will NOT work on JunOS or NX-OS."""
    ifaces   = []
    pattern  = r"interface (\S+)
((?:[ !].*
)*)"
    for m in re.finditer(pattern, config + "
", re.MULTILINE):
        name   = m.group(1)
        body   = m.group(2)
        ip_m   = re.search(r"ip address (\S+) (\S+)", body)
        desc_m = re.search(r"description (.+)", body)
        shut   = "shutdown" in body and "no shutdown" not in body
        ifaces.append({"name": name,
                       "ip":   ip_m.group(1) if ip_m else None,
                       "desc": desc_m.group(1).strip() if desc_m else None,
                       "down": shut})
    return ifaces

# APPROACH 2: AI parser
# Pros: vendor-agnostic, handles unusual formatting, no regex maintenance
# Cons: requires an API call, costs tokens, output may vary slightly

def parse_ifaces_ai(config):
    """Ask Claude to extract interfaces. Works for IOS, NX-OS, JunOS, EOS."""
    resp = client.messages.create(
        model=MODEL, max_tokens=800,
        messages=[{"role": "user",
                   "content": ("Extract all interfaces from this config. "
                               "Return ONLY a JSON array. Each element: "
                               "name, ip_address (null if none), description (null if none), "
                               "is_shutdown (bool). No extra text.

Config:
" + config)}]
    )
    text  = resp.content[0].text
    start = text.find("[")
    end   = text.rfind("]") + 1
    return json.loads(text[start:end]) if start >= 0 and end > start else []

print("=== APPROACH 1: Regex ===")
for i in parse_ifaces_regex(sample_config):
    print(f"  {i['name']:<30} IP: {str(i['ip']):<18} Down: {i['down']}")

print()
print("=== APPROACH 2: AI ===")
for i in parse_ifaces_ai(sample_config):
    print(f"  {i['name']:<30} IP: {str(i.get('ip_address')):<18} Down: {i.get('is_shutdown')}")

print()
print("Identical results for Cisco IOS. The AI parser also works for JunOS and NX-OS.")


---
## Demo 8 - Context Window and Token Cost

The **context window** is the maximum token budget for one API call.
It includes BOTH your input (prompt + config) AND the model output.

This is like MTU - if your payload exceeds it, things break or get truncated.

| Model | Context window |
|-------|----------------|
| Claude Haiku 4.5 / Sonnet 4 / Opus 4 | 200,000 tokens |
| GPT-4o | 128,000 tokens |
| Gemini 1.5 Pro | 1,000,000 tokens |

**The lost-in-the-middle problem:** LLMs recall information better from the
very start and very end of long prompts. Critical config details buried in
the middle of a 500KB dump may be missed. Put the most important context first.


In [ ]:
context_window = 200_000

tasks = [
    ("Quick question",                 "What is OSPF?"),
    ("Single show command",            "show ip bgp summary
..." * 5),
    ("One router config (~500 lines)",  sample_config),
    ("10 router configs at once",       sample_config * 10),
    ("100 router configs at once",      sample_config * 100),
]

print(f"Context window: {context_window:,} tokens (Claude)
")
print(f"{'Task':<38} {'Tokens':>8}  {'% window':>9}  {'Haiku':>10}  {'Sonnet':>10}")
print("-" * 85)

for label, content in tasks:
    count  = client.messages.count_tokens(model=MODEL, messages=[{"role":"user","content":content}])
    tokens = count.input_tokens
    pct    = tokens / context_window * 100
    cost_h = tokens * 0.0000008
    cost_s = tokens * 0.000003
    print(f"{label:<38} {tokens:>8,}  {pct:>8.2f}%    ")

print()
print("Cost optimisation tips (same logic as bandwidth engineering):")
print("  Use Haiku for extraction tasks - 75% cheaper than Sonnet")
print("  Strip comments and blank lines from configs before sending")
print("  Send only the relevant section, not the entire device config")
print("  Batch multiple devices into one request when asking the same question")


---
## Demo 9 - Security: Always Sanitise Before Sending

This is the most important operational rule in this notebook.

When you send a config to the Claude API, it travels over HTTPS to Anthropic servers.
Before sending ANY config to ANY external AI API, always remove:
- Enable passwords and type 5/7/9 secrets
- SNMP community strings (even read-only ones)
- TACACS+ and RADIUS shared keys
- BGP MD5 session passwords
- IPSec pre-shared keys

**Networking analogy:** You would not paste your TACACS+ key into a Teams chat.
Apply the exact same standard to AI APIs. Sanitise locally, then send.


In [ ]:
import re

def sanitise_config(config):
    """Remove all credential values from a router config before sending to any external API.
    Returns: (sanitised_string, list_of_removed_item_descriptions)
    """
    s, removed = config, []
    rules = [
        ("Enable password/secret",   r"(enable\s+(?:password|secret)\s+(?:\d+\s+)?)(\S+)",   r"\1REDACTED"),
        ("Line/interface password",  r"(password\s+)(\S+)",                                    r"\1REDACTED"),
        ("SNMP community string",    r"(snmp-server\s+community\s+)(\S+)",                    r"\1REDACTED"),
        ("TACACS+/RADIUS key",       r"((?:tacacs-server|radius-server)\s+key\s+)(\S+)",      r"\1REDACTED"),
        ("BGP neighbor password",    r"(neighbor\s+\S+\s+password\s+)(\S+)",               r"\1REDACTED"),
        ("IPSec pre-shared key",     r"(pre-shared-key\s+(?:address\s+\S+\s+)?)(\S+)",     r"\1REDACTED"),
    ]
    for desc, pat, repl in rules:
        new_s, count = re.subn(pat, repl, s, flags=re.IGNORECASE)
        if count:
            removed.append(f"{desc} ({count} occurrence{'s' if count > 1 else ''}")
        s = new_s
    return s, removed

dirty = """
hostname edge-router-01
\!
enable secret 5 cMHAE.3dsT4e7Ns9JGrA1
\!
snmp-server community NetworkM0n RO
snmp-server community NetMgmt@2024 RW
\!
router bgp 65001
 neighbor 10.0.0.2 password BGPsecret\!2024
 neighbor 10.0.0.2 remote-as 65002
\!
line vty 0 4
 password VtyP@ss123
"""

clean, removed = sanitise_config(dirty)

print("=== ORIGINAL (contains secrets - do NOT send this) ===")
print(dirty)
print("=== SANITISED (safe to send to any external API) ===")
print(clean)
print("=== REMOVED ===")
for item in removed:
    print(f"  Removed: {item}")
print()
print("Always run sanitise_config() before calling any external API.")


In [ ]:
# Correct production workflow: sanitise first, then analyse
response = client.messages.create(
    model=MODEL,
    max_tokens=800,
    system="You are a senior network security engineer reviewing a sanitised router config.",
    messages=[{
        "role": "user",
        "content": (
            "Review this config for security issues and misconfigurations. "
            "All credential values have been replaced with REDACTED before sending.

"
            + clean
        )
    }]
)

print("=== Security Review of Sanitised Config ===")
print(response.content[0].text)
print()
print("Correct workflow:")
print("  1. Load raw config from your device")
print("  2. Run sanitise_config() to remove all credentials")
print("  3. Send the sanitised version to the AI API")
print("  4. The model still catches misconfigurations - sanitisation does not hide problems")
print("  5. Never let raw credentials leave your network perimeter")


---
## Congratulations - You Completed Chapter 1\!

Here is everything you built and understood in this notebook:

| Demo | Concept | Key analogy |
|------|---------|-------------|
| 1 | LLM API call in 5 lines of Python | ping to the AI |
| 2 | Tokenisation: text becomes integer IDs | Tokens are packets; context window is MTU |
| 3 | Temperature and softmax control output randomness | Temperature is like QoS jitter |
| 4 | Self-attention: transformers understand context | Route Reflector computes relevance weights |
| 5 | Embeddings and cosine similarity: meaning as math | OSPF LSA encodes the topology of meaning |
| 6 | Config review using a system prompt | AI as your 3am senior engineer |
| 7 | Regex vs AI: choosing the right tool | Static route vs routing protocol |
| 8 | Token cost budgeting | Bandwidth planning for API calls |
| 9 | Sanitise before sending to any external API | Never expose credentials to external services |

---

### What is Next?

**Chapter 2 - How LLMs Work in Depth**
Transformer architecture, positional encoding, multi-head attention,
and the complete forward pass implemented step by step in code.

**Chapter 3 - Choosing the Right Model**
Benchmarks, cost analysis, and when to use Haiku vs Sonnet vs Opus
for different network engineering workloads.

---
Built for the book *AI for Networking and Security Engineers* by Edy.
All code is MIT licensed and freely reusable.
